# UAV Swarm Cybersecurity — Hybrid AE-Stacked Ensemble (Binary Attack Detection)

**FYP: Federated-Learning-based UAV Swarm Cybersecurity with Explainable AI**

This notebook is a clean, self-contained, runnable version of the final hybrid model —
no external `.py` imports required, no JSON, one CSV in, all results out.

## ML approaches used in this project (summary for supervisor)

| # | Approach | What it does |
|---|----------|---------------|
| 1 | **Baseline classifiers** (Logistic Regression, Random Forest, MLP) | Traditional supervised models trained centrally on all data, used as a performance baseline. |
| 2 | **Hierarchical Federated Learning (FL)** | Simulates multiple UAVs training locally and sharing only model updates (not raw data) with a central server — privacy-preserving distributed learning. |
| 3 | **Autoencoder (unsupervised)** | Trained only on benign traffic; learns to reconstruct "normal" patterns. Traffic that reconstructs poorly is flagged as anomalous — used for anomaly-based detection. |
| 4 | **Hybrid AE-Stacked Ensemble (3-class)** | Combines the autoencoder's anomaly signal (reconstruction error + latent features) with a 5-model stacking ensemble (Random Forest, XGBoost, LightGBM, Extra Trees, MLP → Logistic Regression meta-learner) to classify benign / DoS attack / Replay. |
| 5 | **Hybrid AE-Stacked Ensemble (binary, this notebook)** | Same architecture as #4, but the target is collapsed to attack-vs-benign — this is the model documented below, achieving the project's headline accuracy. |

**Why binary is the headline result:** independent testing (Random Forest, the full
5-model stack, and a hierarchical 2-stage classifier) all showed that DoS attack and
Replay overlap heavily at the packet-feature level (Replay resends old legitimate
traffic, so its packet signature resembles DoS). This caps 3-class accuracy around
75-76% regardless of model complexity, while binary attack-detection does not suffer
from this overlap and reaches 90%+.


## 1. Imports & configuration

In [ ]:
import json
import copy
import random
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # switch to default backend if running interactively
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix,
    roc_auc_score, precision_score, recall_score,
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


## 2. Load the CSV (with a clear, helpful check)

Set `DATA_PATH` below to point at your dataset. If the notebook can't find the file,
it will tell you exactly what it looked for and where — instead of a confusing
`FileNotFoundError` traceback.

In [ ]:
DATA_PATH = Path("Dataset_T-ITS.csv")  # <-- change this if your CSV has a different name/location
OUTPUT_DIR = Path("uav_fl_results_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find '{DATA_PATH.resolve()}'.\\n"
        f"Current working directory is: {Path.cwd()}\\n"
        f"-> Put Dataset_T-ITS.csv in this same folder, or update DATA_PATH above "
        f"to the full path of your CSV file."
    )

print(f"Found dataset at: {DATA_PATH.resolve()}")
print(f"File size: {DATA_PATH.stat().st_size / 1e6:.2f} MB")


## 3. Load & clean the data

In [ ]:
raw = pd.read_csv(DATA_PATH, dtype=str, low_memory=False)
raw.columns = [str(c).strip() for c in raw.columns]

if "class" not in raw.columns:
    raise ValueError(
        f"Expected a 'class' column in the CSV but found columns: {list(raw.columns)}"
    )

y_all = raw["class"].astype(str).str.strip()
VALID_LABELS = {"benign", "Replay", "DoS attack"}
df = raw.loc[y_all.isin(VALID_LABELS)].copy()
y3 = df.pop("class").astype(str).str.strip()  # original 3-class labels (kept for the secondary report)

X = df.apply(pd.to_numeric, errors="coerce")
X = X.drop(columns=[c for c in ["timestamp_c", "frame.number"] if c in X.columns])   # identifier-like columns
X = X.drop(columns=X.columns[X.isna().all()])                                        # fully-empty columns
nunique = X.nunique(dropna=True)
X = X.drop(columns=nunique[nunique <= 1].index)                                      # constant columns

print(f"Rows after cleaning: {len(X)}   |   Features: {X.shape[1]}")
print(y3.value_counts())

y_bin_all = (y3 != "benign").astype(int)   # 1 = attack (DoS attack or Replay), 0 = benign
print("\\nBinary label counts (0=benign, 1=attack):")
print(y_bin_all.value_counts())


## 4. Train / validation / test split + preprocessing

In [ ]:
Xtv, Xte, ytv3, yte3 = train_test_split(X, y3, test_size=0.20, stratify=y3, random_state=SEED)
Xtr, Xv, ytr3, yv3 = train_test_split(Xtv, ytv3, test_size=0.25, stratify=ytv3, random_state=SEED)

ytr_bin = (ytr3 != "benign").astype(int).values
yv_bin = (yv3 != "benign").astype(int).values
yte_bin = (yte3 != "benign").astype(int).values

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", RobustScaler(quantile_range=(10, 90))),
]).fit(Xtr)

Xtr_p = preprocessor.transform(Xtr).astype("float32")
Xv_p = preprocessor.transform(Xv).astype("float32")
Xte_p = preprocessor.transform(Xte).astype("float32")

print("Train:", Xtr_p.shape, " Val:", Xv_p.shape, " Test:", Xte_p.shape)


## 5. Autoencoder — anomaly-feature extractor

Trained **only on benign traffic** in the training split. Produces two kinds of features for every row: reconstruction error (how "unusual" it looks) and a compressed latent representation.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, n_features),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_autoencoder(Xtr, ytr_bin, epochs=50, patience=8, batch_size=256, weight_decay=1e-5):
    device = torch.device(DEVICE)
    model = Autoencoder(Xtr.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    benign_train = torch.tensor(Xtr[ytr_bin == 0], dtype=torch.float32, device=device)

    best, best_state, wait = np.inf, None, 0
    history = []
    n = benign_train.shape[0]
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        losses = []
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            xb = benign_train[idx]
            opt.zero_grad()
            rec = model(xb)
            loss = loss_fn(rec, xb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            vloss = loss_fn(model(benign_train), benign_train).item()
        history.append({"epoch": epoch + 1, "train_loss": float(np.mean(losses))})
        if vloss < best - 1e-6:
            best, best_state, wait = vloss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()

    def extract(X):
        xt = torch.tensor(X, dtype=torch.float32, device=device)
        with torch.no_grad():
            latent = model.encoder(xt)
            rec = model.decoder(latent)
            err = ((rec - xt) ** 2).mean(dim=1, keepdim=True)
        return np.concatenate([err.cpu().numpy(), latent.cpu().numpy()], axis=1)

    return extract, pd.DataFrame(history)


extract_fn, ae_history = train_autoencoder(Xtr_p, ytr_bin)

Xtr_h = np.concatenate([Xtr_p, extract_fn(Xtr_p)], axis=1)
Xv_h = np.concatenate([Xv_p, extract_fn(Xv_p)], axis=1)
Xte_h = np.concatenate([Xte_p, extract_fn(Xte_p)], axis=1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ae_history.epoch, ae_history.train_loss)
ax.set(xlabel="Epoch", ylabel="MSE (reconstruction loss)", title="Autoencoder training curve")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "autoencoder_training_curve.png", dpi=220)
plt.show()


## 6. Balance the training set (SMOTE)

In [ ]:
sm = SMOTE(random_state=SEED, k_neighbors=5)
Xtr_bal, ytr_bal = sm.fit_resample(Xtr_h, ytr_bin)
print(f"Training rows before SMOTE: {len(ytr_bin)}   after: {len(ytr_bal)}")


## 7. Build & fit the 5-model hybrid stacking ensemble

In [ ]:
rf = RandomForestClassifier(n_estimators=400, max_depth=20, min_samples_leaf=2,
                             max_features="sqrt", class_weight="balanced_subsample",
                             random_state=SEED, n_jobs=-1)
xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                     n_estimators=350, max_depth=6, learning_rate=0.06,
                     subsample=0.85, colsample_bytree=0.85, random_state=SEED,
                     n_jobs=-1, tree_method="hist")
lgbm = LGBMClassifier(objective="binary", n_estimators=350, num_leaves=63,
                       learning_rate=0.06, subsample=0.85, colsample_bytree=0.85,
                       random_state=SEED, n_jobs=-1, verbose=-1)
et = ExtraTreesClassifier(n_estimators=400, max_features="sqrt",
                           class_weight="balanced_subsample", random_state=SEED, n_jobs=-1)
mlp = MLPClassifier(hidden_layer_sizes=(160, 80), alpha=1e-4, batch_size=256,
                     learning_rate_init=8e-4, max_iter=250, early_stopping=True,
                     validation_fraction=0.15, n_iter_no_change=15, random_state=SEED)

meta = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)

stack = StackingClassifier(
    estimators=[("rf", rf), ("xgb", xgb), ("lgbm", lgbm), ("et", et), ("mlp", mlp)],
    final_estimator=meta, stack_method="predict_proba", passthrough=False, n_jobs=-1, cv=5,
)

print("Fitting the 5-model stack... this can take a while on the full dataset.")
stack.fit(Xtr_bal, ytr_bal)
print("Done.")


## 8. Evaluate on the held-out test set

In [ ]:
pred = stack.predict(Xte_h)
prob = stack.predict_proba(Xte_h)[:, 1]

metrics = {
    "accuracy": accuracy_score(yte_bin, pred),
    "balanced_accuracy": balanced_accuracy_score(yte_bin, pred),
    "macro_f1": f1_score(yte_bin, pred, average="macro"),
    "attack_precision": precision_score(yte_bin, pred, pos_label=1),
    "attack_recall": recall_score(yte_bin, pred, pos_label=1),
    "benign_precision": precision_score(yte_bin, pred, pos_label=0),
    "benign_recall": recall_score(yte_bin, pred, pos_label=0),
    "auc": roc_auc_score(yte_bin, prob),
}
print(json.dumps(metrics, indent=2))

pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / "hybrid_binary_test_metrics.csv", index=False)


## 9. Confusion matrix — fixed, clean visualization

This version fixes the earlier formatting problem: text now switches to white on dark cells (so numbers are never invisible), labels are horizontal (no overlapping rotated text), and the figure has enough padding so nothing gets clipped.

In [ ]:
def plot_confusion_clean(y_true, y_pred, labels, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))
    fig, ax = plt.subplots(figsize=(6, 5.5))
    im = ax.imshow(cm, cmap="Blues")

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel("Predicted label", fontsize=12, labelpad=10)
    ax.set_ylabel("True label", fontsize=12, labelpad=10)
    ax.set_title(title, fontsize=13, pad=14)

    threshold = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > threshold else "black"
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                    fontsize=13, color=color, fontweight="bold")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=10)
    fig.tight_layout()
    fig.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()

plot_confusion_clean(
    yte_bin, pred, ["benign", "attack"],
    "Hybrid AE-Stacked Ensemble — Binary Test Confusion Matrix",
    OUTPUT_DIR / "hybrid_binary_confusion_matrix.png",
)


## 10. Comparison chart vs. earlier baselines

In [ ]:
comparison_rows = [
    {"model": "Random Forest (binary, quick baseline)", "value": 0.929},
    {"model": "Hybrid AE-Stack (binary, this notebook)", "value": metrics["accuracy"]},
]
comp_df = pd.DataFrame(comparison_rows)
comp_df.to_csv(OUTPUT_DIR / "hybrid_binary_vs_baseline_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ["#7f9bb8", "#2c6e49"]
ax.bar(comp_df["model"], comp_df["value"], color=colors)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy (test set)")
ax.set_title("Binary attack-detection: baseline vs. hybrid stack")
for i, v in enumerate(comp_df["value"]):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)
plt.xticks(rotation=12, ha="right")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "hybrid_binary_vs_baseline_comparison.png", dpi=220, bbox_inches="tight")
plt.show()


## 11. (Secondary/detailed analysis) — 3-class breakdown

Same test split, original 3-class labels — reported for transparency, since binary is a reframing of the original 3-class task, not a replacement for it.

In [ ]:
le3 = LabelEncoder().fit(y3)
breakdown = pd.DataFrame({
    "true_3class": yte3.values,
    "predicted_binary": np.where(pred == 1, "attack", "benign"),
})
breakdown.to_csv(OUTPUT_DIR / "hybrid_binary_secondary_3class_breakdown.csv", index=False)
print(breakdown.groupby(["true_3class", "predicted_binary"]).size())


## 12. Summary

All outputs are saved under `uav_fl_results_notebook/`:
- `hybrid_binary_test_metrics.csv` — headline numbers
- `hybrid_binary_confusion_matrix.png` — fixed, clean confusion matrix
- `hybrid_binary_vs_baseline_comparison.png` — comparison chart
- `autoencoder_training_curve.png` — training curve
- `hybrid_binary_secondary_3class_breakdown.csv` — transparency detail for the 3-class view

**Headline result:** binary attack-vs-benign detection using the Hybrid AE-Stacked
Ensemble. Report the accuracy/macro-F1 printed in section 8 above — those are the
real numbers from this run, not hardcoded.
